# Insurance Claims Data Science Demo
## Predicting High-Severity Claims -- End-to-End ML Lifecycle on Databricks

**Business Objective:** Predict whether an incoming insurance claim will be high-severity at the time of intake, enabling proactive case management, early reserve allocation, and faster triage.

**Demo Scope (~30 min):**
1. Data Understanding & EDA
2. Target Definition & Leakage Analysis
3. Feature Preparation
4. AutoML + Manual Model Training
5. Evaluation & Business Metrics
6. Explainability (SHAP + Feature Importance)
7. MLflow Experiment Tracking (with plot artifacts)
8. Model Registration (Unity Catalog)
9. Feature Store (centralized features for batch inference)
10. Batch Scoring (Feature Store auto-lookup)
11. Real-Time Serving Endpoint
12. A/B Testing (Champion vs Challenger)
13. Inference Logging & Route Comparison
14. Monitoring & Drift Detection (PSI)
15. Lakehouse Monitoring (automated performance & drift tracking)
16. Scheduled Batch Scoring (Lakeflow Job)

---
**Dataset:** `zurich_lcnc_rfp_catalog.pc_demo.curated_claims_analytics_designer`  
**Target:** `high_severity_claim` = 1 if `incident_severity` in {High, Severe}, else 0  
**Feature Table:** `zurich_lcnc_rfp_catalog.pc_demo.claims_feature_store`  
**Serving Endpoint:** `high-severity-claims-endpoint` (A/B: 80% champion / 20% challenger)  
**Inference Table:** `zurich_lcnc_rfp_catalog.pc_demo.inference_payload`  
**Scored Output:** `zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions`  
**Monitoring Dashboard:** Lakehouse Monitor with daily granularity on scored predictions

In [0]:
%pip install databricks-feature-engineering --quiet
dbutils.library.restartPython()

In [0]:
# ============================================================
# SETUP & IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Set MLflow experiment
EXPERIMENT_NAME = "/Users/stephan.wernz@databricks.com/claims_high_severity_demo"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"✅ MLflow experiment set: {EXPERIMENT_NAME}")
print(f"✅ All imports successful")

In [0]:
# ============================================================
# LOAD DATASET
# ============================================================
TABLE_NAME = "zurich_lcnc_rfp_catalog.pc_demo.curated_claims_analytics_designer"

# Load from Unity Catalog
df_spark = spark.table(TABLE_NAME)
df = df_spark.toPandas()

print(f"📊 Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n📋 Column types:")
print(df.dtypes)
print(f"\n🔍 First 5 rows:")
display(df.head())

In [0]:
# ============================================================
# TARGET DEFINITION
# ============================================================
# Target: high_severity_claim = 1 if incident_severity in ('High', 'Severe'), else 0
df['high_severity_claim'] = df['incident_severity'].apply(
    lambda x: 1 if x in ('High', 'Severe') else 0
)

# Show target distribution
print("🎯 TARGET: high_severity_claim")
print("="*50)
print(f"Definition: incident_severity ∈ {{'High', 'Severe'}} → 1, else → 0")
print(f"\nDistribution:")
target_dist = df['high_severity_claim'].value_counts()
print(target_dist)
print(f"\nClass balance: {target_dist[1]/len(df)*100:.1f}% positive (high severity)")

# Visualize
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df['high_severity_claim'].value_counts().plot(kind='bar', ax=ax[0], color=['#2ecc71', '#e74c3c'])
ax[0].set_title('Target Distribution (Count)', fontsize=12)
ax[0].set_xlabel('High Severity Claim')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)

df['incident_severity'].value_counts().plot(kind='bar', ax=ax[1], color='steelblue')
ax[1].set_title('Original Severity Levels', fontsize=12)
ax[1].set_xlabel('Incident Severity')
ax[1].set_ylabel('Count')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 🔍 Exploratory Data Analysis (EDA)
Before modelling, we explore the dataset to understand distributions, missing values, and relationships between features and the target.

In [0]:
# ============================================================
# EDA: BASIC STATISTICS
# ============================================================
print("📊 Descriptive Statistics (Numeric Features)")
print("="*60)
display(df.describe().round(2))

print(f"\n📊 Categorical Feature Summary:")
for col in ['policy_type', 'region', 'claim_status', 'incident_severity']:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().to_string())

In [0]:
# ============================================================
# EDA: MISSING VALUES
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) > 0:
    print("⚠️ Columns with missing values:")
    display(missing_df)
    
    # Missing values heatmap
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df.isnull().T, cbar=True, cmap='YlOrRd', ax=ax)
    ax.set_title('Missing Values Heatmap', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values found in the dataset!")
    print(f"   Dataset is complete: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [0]:
# ============================================================
# EDA: TARGET DISTRIBUTION BY CATEGORY
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By policy_type
severity_by_policy = df.groupby(['policy_type', 'high_severity_claim']).size().unstack(fill_value=0)
severity_by_policy.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('High Severity Claims by Policy Type', fontsize=12)
axes[0].set_xlabel('Policy Type')
axes[0].set_ylabel('Count')
axes[0].legend(['Not High Severity', 'High Severity'])
axes[0].tick_params(axis='x', rotation=45)

# By region
severity_by_region = df.groupby(['region', 'high_severity_claim']).size().unstack(fill_value=0)
severity_by_region.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('High Severity Claims by Region', fontsize=12)
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Count')
axes[1].legend(['Not High Severity', 'High Severity'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Show severity rates
print("\n📊 High Severity Rate by Policy Type:")
rate_policy = df.groupby('policy_type')['high_severity_claim'].mean().sort_values(ascending=False)
for pt, rate in rate_policy.items():
    print(f"  {pt}: {rate*100:.1f}%")

print("\n📊 High Severity Rate by Region:")
rate_region = df.groupby('region')['high_severity_claim'].mean().sort_values(ascending=False)
for r, rate in rate_region.items():
    print(f"  {r}: {rate*100:.1f}%")

In [0]:
# ============================================================
# EDA: NUMERIC FEATURE DISTRIBUTIONS
# ============================================================
numeric_features = ['premium_amount', 'coverage_limit', 'deductible', 'risk_score', 'reporting_delay_days']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    # Plot distributions split by target
    df[df['high_severity_claim']==0][col].hist(ax=ax, alpha=0.6, label='Not Severe', color='#2ecc71', bins=30)
    df[df['high_severity_claim']==1][col].hist(ax=ax, alpha=0.6, label='High Severity', color='#e74c3c', bins=30)
    ax.set_title(f'{col}', fontsize=11)
    ax.legend()
    ax.set_xlabel(col)

# Remove unused subplot
axes[-1].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Target Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# EDA: CORRELATION MATRIX
# ============================================================
# Select numeric columns relevant for modelling
corr_cols = ['premium_amount', 'coverage_limit', 'deductible', 'risk_score', 
             'reporting_delay_days', 'claim_month', 'claim_year', 'high_severity_claim']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🔗 Top correlations with target (high_severity_claim):")
target_corr = corr_matrix['high_severity_claim'].drop('high_severity_claim').abs().sort_values(ascending=False)
for feat, corr in target_corr.items():
    print(f"  {feat}: {corr:.3f}")

## ⚠️ Leakage Analysis & Feature Selection

Before modelling, we must identify and **exclude** fields that would leak future information into training. These are fields only known **after** claim resolution:

| Field | Reason for Exclusion |
| --- | --- |
| `incident_severity` | Direct target proxy — the label is derived from this column |
| `claim_amount` | Final assessed claim amount, not available at intake |
| `paid_amount` | Settlement/payment outcome |
| `total_payment_amount` | Aggregated post-resolution payments |
| `payment_count` | Number of payments during resolution process |
| `days_to_settle` | Duration from open to close — only known after closure |
| `claim_status` | Final status (e.g., Closed, Settled) — not known at prediction time |
| `claim_close_date` | Only populated after claim resolution |

**Valid features (available at claim intake):**
- `policy_type`, `region` (categorical)
- `premium_amount`, `coverage_limit`, `deductible`, `risk_score` (numeric)
- `reporting_delay_days`, `claim_month`, `claim_year` (temporal/derived)

In [0]:
# ============================================================
# FEATURE PREPARATION
# ============================================================

# Define valid features (available at prediction time)
CATEGORICAL_FEATURES = ['policy_type', 'region']
NUMERIC_FEATURES = ['premium_amount', 'coverage_limit', 'deductible', 
                    'risk_score', 'reporting_delay_days', 'claim_month', 'claim_year']

# Leakage-prone fields — EXCLUDED
LEAKAGE_FIELDS = ['incident_severity', 'claim_amount', 'paid_amount', 
                  'total_payment_amount', 'payment_count', 'days_to_settle',
                  'claim_status', 'claim_close_date']

TARGET = 'high_severity_claim'
ALL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

print("✅ FEATURE SELECTION SUMMARY")
print("="*50)
print(f"\n🎯 Target: {TARGET}")
print(f"\n🟢 Valid features ({len(ALL_FEATURES)}): {ALL_FEATURES}")
print(f"\n🔴 Excluded (leakage): {LEAKAGE_FIELDS}")

# Prepare feature matrix
df_model = df[ALL_FEATURES + [TARGET]].copy()

# Encode categoricals
df_encoded = pd.get_dummies(df_model, columns=CATEGORICAL_FEATURES, drop_first=False)

# Separate features and target
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f"\n📊 Feature matrix shape: {X.shape}")
print(f"   ({X.shape[0]:,} samples × {X.shape[1]} features after one-hot encoding)")
print(f"\n📊 Target distribution:")
print(f"   0 (Not severe): {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"   1 (High severe): {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")

# Train/Test Split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✂️ Train/Test Split:")
print(f"   Training set: {X_train.shape[0]:,} samples")
print(f"   Test set:     {X_test.shape[0]:,} samples")
print(f"   Stratification preserved: Train pos rate = {y_train.mean()*100:.1f}%, Test pos rate = {y_test.mean()*100:.1f}%")

In [0]:
# ============================================================
# SAVE TRAINING DATA TO VOLUME (for AutoML and reproducibility)
# ============================================================
df_spark_from_pandas = spark.createDataFrame(df)
volume_path = "/Volumes/zurich_lcnc_rfp_catalog/pc_demo/raw/training.parquet"
df.to_parquet(volume_path, index=False)
df_spark = spark.read.parquet(volume_path)
display(df_spark)

Databricks data profile. Run in Databricks to view.

## 🤖 Model Training

### Approach
1. **Databricks AutoML** — automated model search with `databricks.automl.classify()`
2. **Manual Candidates** — Logistic Regression and Random Forest logged to MLflow for comparison

All models are tracked in a single MLflow experiment for side-by-side comparison.

In [0]:
# ============================================================
# AUTOML TRAINING
# ============================================================
# Note: databricks.automl requires ML Runtime cluster (not serverless).
# On ML Runtime, the following code runs automated model search.
# We demonstrate the pattern and fall back to manual training below.

try:
    import databricks.automl

    # Prepare Spark DataFrame for AutoML (needs target + valid features only)
    df_automl = df[ALL_FEATURES + [TARGET]].copy()
    df_automl_spark = spark.createDataFrame(df_automl)

    print("Starting Databricks AutoML Classification...")
    print(f"   Target: {TARGET}")
    print(f"   Features: {ALL_FEATURES}")
    print(f"   Primary metric: F1")
    print(f"   Timeout: 5 minutes")
    print("="*50)

    # Run AutoML
    automl_summary = databricks.automl.classify(
        dataset=df_automl_spark,
        target_col=TARGET,
        primary_metric="f1",
        timeout_minutes=5
    )

    print(f"\nAutoML completed!")
    print(f"   Best trial metric (F1): {automl_summary.best_trial.metrics.get('test_f1_score', 'N/A')}")
    print(f"   Best model: {automl_summary.best_trial.model_description}")
    AUTOML_AVAILABLE = True

except (ModuleNotFoundError, ImportError):
    print("NOTE: Databricks AutoML requires an ML Runtime cluster (e.g., DBR 14.3 ML+).")
    print("Serverless compute does not include the AutoML package.")
    print("")
    print("AutoML API pattern (for ML Runtime):")
    print("-" * 45)
    print("  import databricks.automl")
    print("  summary = databricks.automl.classify(")
    print("      dataset=spark_df,")
    print("      target_col='high_severity_claim',")
    print("      primary_metric='f1',")
    print("      timeout_minutes=5")
    print("  )")
    print("-" * 45)
    print("")
    print("Proceeding with manual model training in the next cell...")
    AUTOML_AVAILABLE = False

In [0]:
# ============================================================
# MANUAL MODEL TRAINING (Logged to MLflow)
# ============================================================
import tempfile, os

mlflow.set_experiment(EXPERIMENT_NAME)

def log_model_plots(y_true, y_pred, y_prob, model_name, run):
    """Generate and log ROC + Confusion Matrix plots as MLflow artifacts."""
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
    ax_roc.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'{model_name} (AUC={auc_val:.3f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title(f'ROC Curve - {model_name}')
    ax_roc.legend(loc='lower right')
    ax_roc.grid(True, alpha=0.3)
    plt.tight_layout()
    mlflow.log_figure(fig_roc, "plots/roc_curve.png")
    plt.close(fig_roc)

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                xticklabels=['Not Severe', 'High Severe'],
                yticklabels=['Not Severe', 'High Severe'])
    ax_cm.set_xlabel('Predicted')
    ax_cm.set_ylabel('Actual')
    ax_cm.set_title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    mlflow.log_figure(fig_cm, "plots/confusion_matrix.png")
    plt.close(fig_cm)

# --- Model 1: Logistic Regression ---
with mlflow.start_run(run_name="LogisticRegression_baseline") as run_lr:
    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr.fit(X_train, y_train)
    
    y_pred_lr = lr.predict(X_test)
    y_prob_lr = lr.predict_proba(X_test)[:, 1]
    
    metrics_lr = {
        "accuracy": accuracy_score(y_test, y_pred_lr),
        "f1": f1_score(y_test, y_pred_lr),
        "precision": precision_score(y_test, y_pred_lr),
        "recall": recall_score(y_test, y_pred_lr),
        "auc": roc_auc_score(y_test, y_prob_lr)
    }
    
    mlflow.log_params({"model_type": "LogisticRegression", "max_iter": 1000, "class_weight": "balanced"})
    mlflow.log_metrics(metrics_lr)
    signature = infer_signature(X_train, y_pred_lr)
    mlflow.sklearn.log_model(lr, "model", signature=signature, input_example=X_train.iloc[:3])
    
    # Log plots as artifacts
    log_model_plots(y_test, y_pred_lr, y_prob_lr, "Logistic Regression", run_lr)
    
    lr_run_id = run_lr.info.run_id
    print(f"Logistic Regression | F1={metrics_lr['f1']:.4f} | AUC={metrics_lr['auc']:.4f} | Recall={metrics_lr['recall']:.4f}")
    print(f"   Artifacts logged: plots/roc_curve.png, plots/confusion_matrix.png")

# --- Model 2: Random Forest ---
with mlflow.start_run(run_name="RandomForest_comparison") as run_rf:
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    
    y_pred_rf = rf.predict(X_test)
    y_prob_rf = rf.predict_proba(X_test)[:, 1]
    
    metrics_rf = {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "f1": f1_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "recall": recall_score(y_test, y_pred_rf),
        "auc": roc_auc_score(y_test, y_prob_rf)
    }
    
    mlflow.log_params({"model_type": "RandomForest", "n_estimators": 200, "max_depth": 10, "class_weight": "balanced"})
    mlflow.log_metrics(metrics_rf)
    signature = infer_signature(X_train, y_pred_rf)
    mlflow.sklearn.log_model(rf, "model", signature=signature, input_example=X_train.iloc[:3])
    
    # Log plots as artifacts
    log_model_plots(y_test, y_pred_rf, y_prob_rf, "Random Forest", run_rf)
    
    rf_run_id = run_rf.info.run_id
    print(f"Random Forest      | F1={metrics_rf['f1']:.4f} | AUC={metrics_rf['auc']:.4f} | Recall={metrics_rf['recall']:.4f}")
    print(f"   Artifacts logged: plots/roc_curve.png, plots/confusion_matrix.png")

# --- Compare ---
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
comparison = pd.DataFrame({
    'Logistic Regression': metrics_lr,
    'Random Forest': metrics_rf
}).round(4)
display(comparison)

# Select best model
if metrics_rf['f1'] >= metrics_lr['f1']:
    best_model = rf
    best_model_name = "Random Forest"
    best_run_id = rf_run_id
    y_pred_best = y_pred_rf
    y_prob_best = y_prob_rf
    best_metrics = metrics_rf
else:
    best_model = lr
    best_model_name = "Logistic Regression"
    best_run_id = lr_run_id
    y_pred_best = y_pred_lr
    y_prob_best = y_prob_lr
    best_metrics = metrics_lr

print(f"\nBest manual model: {best_model_name} (F1={best_metrics['f1']:.4f})")
print(f"   Run ID: {best_run_id}")

## 📊 Model Evaluation

**Business Context:** For high-severity claim prediction, **recall** is critical — missing a truly severe claim means delayed response, increased costs, and poor customer outcomes. We accept a precision trade-off to catch more genuine severe cases.

Key metrics shown:
- **AUC/ROC Curve** — ranking performance across all thresholds
- **Confusion Matrix** — TP/FP/FN/TN breakdown
- **Classification Report** — precision, recall, F1 per class

In [0]:
# ============================================================
# MODEL EVALUATION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
auc_val = roc_auc_score(y_test, y_prob_best)

axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'{best_model_name} (AUC={auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=12)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Not Severe', 'High Severe'],
            yticklabels=['Not Severe', 'High Severe'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix', fontsize=12)

# --- Metrics Summary ---
metrics_display = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC'],
    'Value': [best_metrics['accuracy'], best_metrics['precision'], 
              best_metrics['recall'], best_metrics['f1'], best_metrics['auc']]
})
axes[2].axis('off')
table = axes[2].table(
    cellText=[[f"{row['Metric']}", f"{row['Value']:.4f}"] for _, row in metrics_display.iterrows()],
    colLabels=['Metric', 'Score'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.8)
axes[2].set_title(f'Best Model: {best_model_name}', fontsize=12, pad=20)

plt.tight_layout()
plt.show()

# --- Log evaluation dashboard to MLflow ---
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_figure(fig, "plots/evaluation_dashboard.png")
    print("Logged: plots/evaluation_dashboard.png")
plt.close(fig)

# Classification Report
print(f"\nClassification Report ({best_model_name}):")
print("="*50)
print(classification_report(y_test, y_pred_best, target_names=['Not Severe', 'High Severity']))

print("\nBusiness Interpretation:")
print(f"   Recall = {best_metrics['recall']:.1%} -- We catch {best_metrics['recall']:.1%} of truly severe claims")
print(f"   Precision = {best_metrics['precision']:.1%} -- {best_metrics['precision']:.1%} of flagged claims are actually severe")
print(f"   False negatives (missed severe): {cm[1][0]} claims")
print(f"   False positives (false alarms): {cm[0][1]} claims")

## 🔍 Explainability

Understanding **why** the model predicts high severity is critical for:
- Claims adjuster trust and adoption
- Regulatory transparency
- Identifying actionable risk drivers

We show:
1. **Feature importance** from the tree-based model
2. **SHAP values** for global and local explanations

In [0]:
# ============================================================
# EXPLAINABILITY: Feature Importance + SHAP
# ============================================================

# --- Feature Importance (from Random Forest) ---
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    importances = np.abs(best_model.coef_[0])

feat_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig_imp, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=feat_imp.head(15), x='Importance', y='Feature', palette='viridis', ax=ax)
ax.set_title(f'Top 15 Feature Importances ({best_model_name})', fontsize=14)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

# Log feature importance plot to MLflow
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_figure(fig_imp, "plots/feature_importance.png")
    print("Logged: plots/feature_importance.png")
plt.close(fig_imp)

print("\nTop 5 Predictive Features:")
for i, row in feat_imp.head(5).iterrows():
    print(f"   {row['Feature']}: {row['Importance']:.4f}")

# --- SHAP Values ---
try:
    import shap
    
    print("\nComputing SHAP values for global explanation...")
    
    X_shap = X_test.iloc[:200]
    
    if hasattr(best_model, 'feature_importances_'):
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_shap)
        if isinstance(shap_values, list):
            shap_vals = shap_values[1]
        else:
            shap_vals = shap_values
    else:
        explainer = shap.LinearExplainer(best_model, X_train)
        shap_vals = explainer.shap_values(X_shap)
    
    # SHAP Summary Plot
    shap.summary_plot(shap_vals, X_shap, max_display=13, plot_size=(10, 8), show=False)
    fig_shap = plt.gcf()
    fig_shap.suptitle('SHAP Feature Impact on High Severity Prediction', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Log SHAP plot to MLflow
    with mlflow.start_run(run_id=best_run_id):
        mlflow.log_figure(fig_shap, "plots/shap_summary.png")
        print("Logged: plots/shap_summary.png")
    plt.close(fig_shap)
    
    # SHAP bar plot (mean absolute SHAP values)
    shap.summary_plot(shap_vals, X_shap, plot_type="bar", max_display=13, plot_size=(10, 6), show=False)
    fig_shap_bar = plt.gcf()
    fig_shap_bar.suptitle('Mean |SHAP| Value per Feature', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
    
    with mlflow.start_run(run_id=best_run_id):
        mlflow.log_figure(fig_shap_bar, "plots/shap_bar.png")
        print("Logged: plots/shap_bar.png")
    plt.close(fig_shap_bar)
    
    print("\nSHAP explanation generated:")
    print("   Red = high feature value, Blue = low feature value")
    print("   X-axis = impact on prediction (positive pushes toward high severity)")
    
except ImportError:
    print("SHAP not available -- install with: %pip install shap")
except Exception as e:
    print(f"SHAP computation error: {e}")
    print("   Feature importance plot above provides the explainability view.")

## 📝 MLflow Experiment Tracking

All model runs are tracked in a single MLflow experiment. This enables:
- **Reproducibility** — every run captures parameters, metrics, and artifacts
- **Comparison** — side-by-side metric comparison across candidates
- **Lineage** — full audit trail from data to model

In [0]:
# ============================================================
# MLFLOW EXPERIMENT TRACKING
# ============================================================

# Retrieve all runs from our experiment
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.f1 DESC"]
)

# Show key columns
display_cols = ['run_id', 'experiment_id', 'status', 'start_time',
                'params.model_type', 'metrics.accuracy', 'metrics.f1', 
                'metrics.precision', 'metrics.recall', 'metrics.auc']
available_cols = [c for c in display_cols if c in runs_df.columns]

print(f"📊 MLflow Experiment: {EXPERIMENT_NAME}")
print(f"   Total runs: {len(runs_df)}")
print(f"\n🏆 Runs ranked by F1 score:")
display(runs_df[available_cols].head(10))

# Visualize metric comparison
if 'params.model_type' in runs_df.columns:
    manual_runs = runs_df[runs_df['params.model_type'].notna()].copy()
    if len(manual_runs) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        metrics_to_plot = ['metrics.f1', 'metrics.recall', 'metrics.auc', 'metrics.precision']
        plot_data = manual_runs[['params.model_type'] + [m for m in metrics_to_plot if m in manual_runs.columns]].set_index('params.model_type')
        plot_data.columns = [c.replace('metrics.', '') for c in plot_data.columns]
        plot_data.plot(kind='bar', ax=ax, colormap='Set2')
        ax.set_title('Model Comparison — MLflow Tracked Metrics', fontsize=12)
        ax.set_xlabel('Model')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1)
        ax.legend(loc='lower right')
        ax.tick_params(axis='x', rotation=0)
        plt.tight_layout()
        plt.show()

print("\n💡 MLflow UI: Navigate to Experiments in the left sidebar to explore full run details, artifacts, and model artifacts.")

## 📦 Model Registration (Unity Catalog)

We register the best model to Unity Catalog for:
- **Versioning** — track model iterations
- **Governance** — access control via Unity Catalog
- **Deployment** — promote to serving or batch scoring
- **Lineage** — connect model to training data and experiment

In [0]:
# ============================================================
# MODEL REGISTRATION TO UNITY CATALOG
# ============================================================
import mlflow
mlflow.set_registry_uri("databricks-uc")

# Model name in Unity Catalog (3-level namespace)
MODEL_NAME = "zurich_lcnc_rfp_catalog.pc_demo.high_severity_claim_model"

# Register best manual model
model_uri = f"runs:/{best_run_id}/model"

print(f"📦 Registering model to Unity Catalog...")
print(f"   Model URI: {model_uri}")
print(f"   Registry name: {MODEL_NAME}")
print(f"   Model type: {best_model_name}")

try:
    # Register the model
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )
    
    print(f"\n✅ Model registered successfully!")
    print(f"   Name: {model_version.name}")
    print(f"   Version: {model_version.version}")
    print(f"   Status: {model_version.status}")
    print(f"\n📌 Model reference: models:/{MODEL_NAME}/{model_version.version}")
    
    # Set alias for the champion model
    from mlflow import MlflowClient
    client = MlflowClient()
    client.set_registered_model_alias(MODEL_NAME, "champion", model_version.version)
    print(f"   Alias 'champion' set for version {model_version.version}")
    
except Exception as e:
    print(f"⚠️ Registration note: {e}")
    print("   This may occur if the schema doesn't exist or permissions are insufficient.")
    print("   In production, ensure the UC schema exists and you have CREATE MODEL permissions.")
    model_version = None

## 🗄️ Feature Store (Unity Catalog)

We publish the engineered features to a **Unity Catalog Feature Table** so that:
- Features are defined once and reused across training, batch scoring, and future models
- Batch inference only requires **primary keys** — features are automatically looked up
- Full lineage tracks which models consume which features
- Feature freshness and quality are centrally governed

**Approach for batch inference:**
1. Create a feature table with `claim_id` as primary key
2. Log the model with `FeatureLookup` metadata (model "remembers" where its features live)
3. At scoring time, call `fe.score_batch(model_uri, df_with_keys_only)` — features are auto-joined

In [0]:
# ============================================================
# FEATURE STORE: Create Feature Table in Unity Catalog
# ============================================================
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql import functions as F

fe = FeatureEngineeringClient()

FEATURE_TABLE = "zurich_lcnc_rfp_catalog.pc_demo.claims_feature_store"

# Prepare features DataFrame: claim_id (primary key) + all engineered features
feature_columns = list(X_train.columns)
features_df = pd.concat([df[['claim_id']].astype(str), X], axis=1)

# Cast uint8 columns (from get_dummies) to int32 for Arrow compatibility
uint8_cols = features_df.select_dtypes(include=['uint8']).columns
features_df[uint8_cols] = features_df[uint8_cols].astype('int32')

features_spark = spark.createDataFrame(features_df).withColumn('claim_id', F.col('claim_id').cast('string'))

print(f"Creating feature table: {FEATURE_TABLE}")
print(f"="*60)
print(f"   Primary key: claim_id")
print(f"   Features: {len(feature_columns)} columns")
print(f"   Rows: {len(features_df):,}")
print()

# Create (or overwrite) the feature table
try:
    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["claim_id"],
        df=features_spark,
        description="Engineered features for high-severity claim prediction. "
                    "Includes numeric risk features and one-hot encoded categoricals."
    )
    print("Feature table created:", FEATURE_TABLE)
except Exception as e:
    if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
        # Table exists - overwrite data
        fe.write_table(
            name=FEATURE_TABLE,
            df=features_spark,
            mode="overwrite"
        )
        print("Feature table updated (overwrite):", FEATURE_TABLE)
    else:
        raise e

print(f"\nFeature columns:")
for col in feature_columns:
    print(f"   - {col}")

print(f"\nLineage: This table is now tracked in Unity Catalog.")
print(f"   View in Catalog Explorer -> {FEATURE_TABLE} -> Lineage tab")

In [0]:
# ============================================================
# FEATURE STORE: Log Model with Feature Lookups
# ============================================================
# This packages the model with metadata telling it WHERE to find
# features at inference time. score_batch() uses this to auto-join.
# ============================================================

print("Logging model with Feature Store lookups...")
print("="*60)

# Define feature lookups — tells the model which table + columns to use
feature_lookups = [
    FeatureLookup(
        table_name=FEATURE_TABLE,
        feature_names=feature_columns,
        lookup_key="claim_id"
    )
]

# Create a training set reference (model needs to know the label too)
# Cast claim_id to string to match the feature table primary key type
training_pdf = df[['claim_id', TARGET]].copy()
training_pdf['claim_id'] = training_pdf['claim_id'].astype(str)
training_df = spark.createDataFrame(training_pdf)

training_set = fe.create_training_set(
    df=training_df,
    feature_lookups=feature_lookups,
    label=TARGET,
    exclude_columns=['claim_id']
)

# Log model WITH feature store metadata
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="RandomForest_FeatureStore") as fs_run:
    fe.log_model(
        model=best_model,
        artifact_path="fs_model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=MODEL_NAME
    )
    mlflow.log_params({"model_type": "RandomForest", "feature_store": True})
    mlflow.log_metrics(best_metrics)
    fs_run_id = fs_run.info.run_id

print(f"\nModel logged with Feature Store lookups")
print(f"   Run ID: {fs_run_id}")
print(f"   Registered to: {MODEL_NAME}")
print(f"   Feature table: {FEATURE_TABLE}")
print(f"\n   At batch scoring time, only 'claim_id' is needed --")
print(f"   all {len(feature_columns)} features are automatically looked up.")

## 🚀 Batch Scoring (via Feature Store)

With the model packaged with Feature Store lookups, batch scoring becomes trivial:
1. Provide a DataFrame with **only the primary keys** (`claim_id`)
2. `fe.score_batch()` automatically joins all features from the feature table
3. Write predictions to a Delta table for downstream consumption

This eliminates manual feature engineering at inference time and guarantees feature consistency between training and scoring.

In [0]:
# ============================================================
# BATCH SCORING via Feature Store
# ============================================================
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

print("Batch Scoring Pipeline (Feature Store)")
print("="*60)

# --- Only need primary keys for scoring! ---
scoring_pdf = df[['claim_id']].copy()
scoring_pdf['claim_id'] = scoring_pdf['claim_id'].astype(str)
scoring_keys = spark.createDataFrame(scoring_pdf)
print(f"   Input: {scoring_keys.count():,} claim_ids (keys only)")
print(f"   Features will be auto-looked up from: {FEATURE_TABLE}")

# --- Score using Feature Store (auto-joins features) ---
try:
    # Use the latest registered version with feature store metadata
    fs_model_uri = f"models:/{MODEL_NAME}@champion"
    
    predictions_df = fe.score_batch(
        model_uri=fs_model_uri,
        df=scoring_keys
    )
    
    print(f"\nFeature Store score_batch() succeeded!")
    print(f"   Model: {fs_model_uri}")
    print(f"   Features automatically joined from feature table")
    
    # Rename prediction column and add metadata
    predictions_df = predictions_df.withColumnRenamed("prediction", "predicted_high_severity")
    
except Exception as e:
    print(f"\nFeature Store score_batch note: {e}")
    print("   Falling back to direct model scoring...")
    
    # Fallback: manual scoring (same result, just without auto-lookup)
    try:
        loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
    except Exception:
        loaded_model = best_model
    
    X_score = df_encoded.drop(columns=[TARGET])
    df['predicted_high_severity'] = loaded_model.predict(X_score)
    df['severity_probability'] = loaded_model.predict_proba(X_score)[:, 1]
    
    scored_output = df[['claim_id', 'policy_id', 'policy_type', 'region', 'risk_score',
                        'premium_amount', 'coverage_limit', 'deductible',
                        'predicted_high_severity', 'severity_probability']].copy()
    predictions_df = spark.createDataFrame(scored_output)

# --- Show sample predictions ---
print(f"\nSample predictions:")
display(predictions_df.limit(10))

# --- Write to Delta table ---
OUTPUT_TABLE = "zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions"

from pyspark.sql import functions as F

scored_final = predictions_df.withColumn("scored_at", F.current_timestamp()) \
                             .withColumn("model_version", F.lit(best_model_name))

try:
    scored_final.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)
    row_count = spark.table(OUTPUT_TABLE).count()
    print(f"\nPredictions written to: {OUTPUT_TABLE}")
    print(f"   Rows: {row_count:,}")
except Exception as e:
    print(f"\nWrite note: {e}")
    display(predictions_df.limit(5))

print(f"\nKey advantage: scoring code has NO feature engineering logic.")
print(f"   Features are managed centrally in {FEATURE_TABLE}")

## Real-Time Serving Endpoint

Deploy the registered model as a REST API for real-time scoring. This enables:
- Low-latency predictions at claim intake
- Integration with claims management systems via REST calls
- Auto-scaling based on traffic

In [0]:
# ============================================================
# DEPLOY MODEL SERVING ENDPOINT
# ============================================================
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)
import time

w = WorkspaceClient()

ENDPOINT_NAME = "high-severity-claims-endpoint"
MODEL_NAME = "zurich_lcnc_rfp_catalog.pc_demo.high_severity_claim_model"
MODEL_VERSION = 1  # Use the version registered earlier

print(f"Deploying serving endpoint: {ENDPOINT_NAME}")
print(f"   Model: {MODEL_NAME} v{MODEL_VERSION}")
print("=" * 50)

try:
    # Create or update the endpoint
    endpoint = w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version=str(MODEL_VERSION),
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                )
            ],
        ),
    )
    print(f"\nEndpoint created successfully!")
    print(f"   State: {endpoint.state.ready}")

except Exception as e:
    if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
        print(f"Endpoint '{ENDPOINT_NAME}' already exists. Updating...")
        try:
            endpoint = w.serving_endpoints.update_config_and_wait(
                name=ENDPOINT_NAME,
                served_entities=[
                    ServedEntityInput(
                        entity_name=MODEL_NAME,
                        entity_version=str(MODEL_VERSION),
                        workload_size="Small",
                        scale_to_zero_enabled=True,
                    )
                ],
            )
            print(f"   Updated successfully! State: {endpoint.state.ready}")
        except Exception as update_err:
            print(f"   Update note: {update_err}")
    else:
        print(f"Endpoint creation note: {e}")
        print("   Ensure you have serving endpoint permissions in this workspace.")

In [0]:
# ============================================================
# TEST SERVING ENDPOINT (sample inference request)
# ============================================================
import json

# Sample claims for real-time scoring
sample_claims = X_test.head(3).to_dict(orient='records')

print("Testing endpoint with sample claims...")
print("=" * 50)
print(f"\nSample request payload ({len(sample_claims)} claims):")
print(json.dumps({"dataframe_records": sample_claims[:1]}, indent=2, default=str))

try:
    response = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        dataframe_records=sample_claims
    )
    
    print(f"\nEndpoint response:")
    print(f"   Predictions: {response.predictions}")
    print(f"\nReal-time scoring operational!")
    
except Exception as e:
    error_msg = str(e)
    if "not found" in error_msg.lower() or "NOT_FOUND" in error_msg:
        print(f"\nEndpoint '{ENDPOINT_NAME}' not yet ready (may still be provisioning).")
        print("   Typical warm-up time: 5-10 minutes for scale-to-zero endpoints.")
    elif "not ready" in error_msg.lower():
        print(f"\nEndpoint is provisioning. Check status in the Serving UI.")
    else:
        print(f"\nQuery note: {e}")
    
    print("\nSample curl command (once endpoint is ready):")
    print(f'  curl -X POST https://<workspace-url>/serving-endpoints/{ENDPOINT_NAME}/invocations \\')
    print(f'    -H "Authorization: Bearer <token>" \\')
    print(f'    -H "Content-Type: application/json" \\')
    print(f'    -d \'{{"dataframe_records": {json.dumps(sample_claims[:1], default=str)}}}\'')

## A/B Testing: Champion vs Challenger

We train a **GradientBoosting challenger** model, register it as version 2, then split endpoint traffic:
- **80%** to the champion (Random Forest v1)
- **20%** to the challenger (GradientBoosting v2)

This validates whether the new model improves outcomes on live traffic before a full rollover.

In [0]:
# ============================================================
# TRAIN & REGISTER CHALLENGER MODEL (v2)
# ============================================================
from sklearn.ensemble import GradientBoostingClassifier

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="GradientBoosting_challenger") as run_gb:
    gb = GradientBoostingClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=42
    )
    gb.fit(X_train, y_train)

    y_pred_gb = gb.predict(X_test)
    y_prob_gb = gb.predict_proba(X_test)[:, 1]

    metrics_gb = {
        "accuracy": accuracy_score(y_test, y_pred_gb),
        "f1": f1_score(y_test, y_pred_gb),
        "precision": precision_score(y_test, y_pred_gb),
        "recall": recall_score(y_test, y_pred_gb),
        "auc": roc_auc_score(y_test, y_prob_gb),
    }

    mlflow.log_params({
        "model_type": "GradientBoosting",
        "n_estimators": 150,
        "max_depth": 5,
        "learning_rate": 0.1,
    })
    mlflow.log_metrics(metrics_gb)
    signature = infer_signature(X_train, y_pred_gb)
    mlflow.sklearn.log_model(gb, "model", signature=signature, input_example=X_train.iloc[:3])

    gb_run_id = run_gb.info.run_id

print("Challenger model trained:")
print(f"   GradientBoosting | F1={metrics_gb['f1']:.4f} | AUC={metrics_gb['auc']:.4f} | Recall={metrics_gb['recall']:.4f}")
print(f"\nCompared to champion (Random Forest v1):")
print(f"   Random Forest     | F1={best_metrics['f1']:.4f} | AUC={best_metrics['auc']:.4f} | Recall={best_metrics['recall']:.4f}")

# Register as version 2
mlflow.set_registry_uri("databricks-uc")
challenger_uri = f"runs:/{gb_run_id}/model"

try:
    challenger_version = mlflow.register_model(
        model_uri=challenger_uri,
        name=MODEL_NAME
    )
    print(f"\nChallenger registered: {MODEL_NAME} v{challenger_version.version}")

    from mlflow import MlflowClient
    client = MlflowClient()
    client.set_registered_model_alias(MODEL_NAME, "challenger", challenger_version.version)
    print(f"   Alias 'challenger' set for version {challenger_version.version}")
    CHALLENGER_VERSION = challenger_version.version
except Exception as e:
    print(f"\nRegistration note: {e}")
    CHALLENGER_VERSION = "2"

In [0]:
# ============================================================
# A/B TEST: Update endpoint with traffic split
# ============================================================
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    TrafficConfig,
    Route,
)

print("Configuring A/B traffic split on serving endpoint...")
print("=" * 50)
print(f"   Champion  (v1): 80% traffic")
print(f"   Challenger (v{CHALLENGER_VERSION}): 20% traffic")
print()

try:
    endpoint = w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=MODEL_NAME,
                entity_version="1",
                workload_size="Small",
                scale_to_zero_enabled=True,
                name="champion-v1",
            ),
            ServedEntityInput(
                entity_name=MODEL_NAME,
                entity_version=str(CHALLENGER_VERSION),
                workload_size="Small",
                scale_to_zero_enabled=True,
                name="challenger-v2",
            ),
        ],
        traffic_config=TrafficConfig(
            routes=[
                Route(served_model_name="champion-v1", traffic_percentage=80),
                Route(served_model_name="challenger-v2", traffic_percentage=20),
            ]
        ),
    )
    print(f"\nA/B test deployed!")
    print(f"   Endpoint state: {endpoint.state.ready}")
    print(f"\nTraffic routing:")
    print(f"   champion-v1    -> 80% (Random Forest)")
    print(f"   challenger-v2  -> 20% (GradientBoosting)")
    print(f"\nMonitor prediction distributions per route to evaluate the challenger.")
    print(f"Once challenger proves superior, promote to 100% with:")
    print(f"   client.set_registered_model_alias(MODEL_NAME, 'champion', {CHALLENGER_VERSION})")

except Exception as e:
    print(f"A/B update note: {e}")
    print("   Ensure the endpoint exists and both model versions are registered.")

## Inference Logging for A/B Comparison

We enable **AI Gateway inference tables** on the endpoint to automatically log every request/response with the served route. This lets us:
- Compare prediction distributions between champion and challenger
- Measure latency per route
- Track volume and error rates
- Build a ground-truth join for offline evaluation

In [0]:
# ============================================================
# ENABLE AI GATEWAY INFERENCE TABLE (REST API)
# ============================================================
import requests
import json

print("Enabling AI Gateway inference table on endpoint...")
print("=" * 50)

INFERENCE_TABLE_CATALOG = "zurich_lcnc_rfp_catalog"
INFERENCE_TABLE_SCHEMA = "pc_demo"

# Use workspace client to get host and token
host = w.config.host.rstrip("/")
token = w.config.token

# Enable inference table via REST API
payload = {
    "ai_gateway_config": {
        "inference_table_config": {
            "catalog_name": INFERENCE_TABLE_CATALOG,
            "schema_name": INFERENCE_TABLE_SCHEMA,
            "enabled": True
        }
    }
}

try:
    resp = requests.put(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/ai-gateway",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=payload
    )
    
    if resp.status_code == 200:
        print(f"\nInference table enabled!")
        print(f"   Catalog: {INFERENCE_TABLE_CATALOG}")
        print(f"   Schema:  {INFERENCE_TABLE_SCHEMA}")
        print(f"   Table:   {INFERENCE_TABLE_CATALOG}.{INFERENCE_TABLE_SCHEMA}.`{ENDPOINT_NAME}_payload`")
        print(f"\n   Every request/response is now logged with:")
        print(f"   - Timestamp, request payload, response payload")
        print(f"   - Served model name (champion-v1 or challenger-v2)")
        print(f"   - Latency, status code")
    else:
        print(f"   Response ({resp.status_code}): {resp.text}")
        print(f"\n   If AI Gateway is not available, inference logging can be")
        print(f"   configured manually in the Serving UI under 'Inference tables'.")

except Exception as e:
    print(f"Inference table note: {e}")
    print("   Ensure the endpoint exists and AI Gateway is enabled.")

In [0]:
# ============================================================
# SEND TRAFFIC TO POPULATE INFERENCE LOGS
# ============================================================
import json
import time

print("Sending 20 sample requests to populate inference logs...")
print("=" * 50)

# Send requests to generate logs across both routes
sample_batch = X_test.head(20).to_dict(orient='records')
route_counts = {"champion-v1": 0, "challenger-v2": 0, "unknown": 0}

for i in range(0, len(sample_batch), 2):
    batch = sample_batch[i:i+2]
    try:
        response = w.serving_endpoints.query(
            name=ENDPOINT_NAME,
            dataframe_records=batch
        )
        # Count based on expected 80/20 split
        route_counts["unknown"] += len(batch)  # SDK doesn't expose route in response
    except Exception as e:
        pass
    time.sleep(0.5)  # Avoid rate limiting

print(f"\nSent {len(sample_batch)} requests to endpoint.")
print(f"Inference logs will be available in ~2-5 minutes.")
print(f"\nExpected distribution (80/20 split):")
print(f"   champion-v1:    ~{int(len(sample_batch)*0.8)} requests")
print(f"   challenger-v2:  ~{int(len(sample_batch)*0.2)} requests")

In [0]:
# ============================================================
# QUERY INFERENCE TABLE TO COMPARE ROUTES
# ============================================================
import time

# Inference table (confirmed in UI)
INFERENCE_TABLE = "zurich_lcnc_rfp_catalog.pc_demo.inference_payload"

print(f"Querying inference table: {INFERENCE_TABLE}")
print("=" * 50)
print("\nNote: Inference logs have a ~2-5 min delay before appearing.")
print("If the table is empty, wait and re-run this cell.\n")

try:
    # Query the inference table
    inference_df = spark.sql(f"""
        SELECT
            databricks_request_id,
            request_date,
            request_time,
            status_code,
            execution_duration_ms,
            served_entity_id,
            request,
            response
        FROM {INFERENCE_TABLE}
        WHERE request_date >= current_date()
        ORDER BY request_time DESC
    """)
    
    log_count = inference_df.count()
    print(f"Total inference logs today: {log_count}")
    
    if log_count > 0:
        # Route-level comparison
        print("\n--- ROUTE COMPARISON ---")
        route_stats = spark.sql(f"""
            SELECT
                served_entity_id AS route,
                COUNT(*) AS request_count,
                ROUND(AVG(execution_duration_ms), 1) AS avg_latency_ms,
                ROUND(MIN(execution_duration_ms), 1) AS min_latency_ms,
                ROUND(MAX(execution_duration_ms), 1) AS max_latency_ms,
                ROUND(PERCENTILE(execution_duration_ms, 0.95), 1) AS p95_latency_ms
            FROM {INFERENCE_TABLE}
            WHERE request_date >= current_date()
            GROUP BY served_entity_id
            ORDER BY request_count DESC
        """)
        display(route_stats)
        
        # Traffic split validation
        print("\n--- TRAFFIC SPLIT VALIDATION ---")
        traffic_split = spark.sql(f"""
            SELECT
                served_entity_id AS route,
                COUNT(*) AS requests,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS traffic_pct
            FROM {INFERENCE_TABLE}
            WHERE request_date >= current_date()
            GROUP BY served_entity_id
        """)
        display(traffic_split)
        
        # Show sample logs
        print("\n--- SAMPLE INFERENCE LOGS ---")
        display(inference_df.limit(5))
    else:
        print("No logs yet. Inference table takes 2-5 minutes to populate.")
        print("Re-run this cell after sending more traffic.")

except Exception as e:
    error_msg = str(e)
    if "TABLE_OR_VIEW_NOT_FOUND" in error_msg or "not found" in error_msg.lower():
        print(f"Table {INFERENCE_TABLE} not yet created.")
        print("Inference tables are auto-created after the first request.")
        print("Wait 2-5 minutes after enabling and sending traffic, then re-run.")
    else:
        print(f"Query note: {e}")

## 📊 Monitoring & Drift Detection

**Why monitor?** Model performance degrades over time as data distributions shift. We need to detect:
- **Data drift** — input feature distributions change
- **Prediction drift** — model output patterns shift
- **Performance drift** — actual accuracy/recall degrades

**Approach:** We compute the Population Stability Index (PSI) to compare training vs. scoring distributions. PSI > 0.2 signals significant drift requiring investigation.

**Retraining triggers:**
- PSI > 0.2 on any key feature
- F1 drops below 0.70 on labelled validation data
- Monthly scheduled retraining regardless

In [0]:
# ============================================================
# MONITORING & DRIFT DETECTION
# ============================================================

def calculate_psi(reference, current, bins=10):
    """Calculate Population Stability Index between two distributions."""
    # Bin the reference distribution
    breakpoints = np.linspace(min(reference.min(), current.min()), 
                              max(reference.max(), current.max()), bins + 1)
    
    ref_counts = np.histogram(reference, bins=breakpoints)[0] + 1  # smoothing
    cur_counts = np.histogram(current, bins=breakpoints)[0] + 1  # smoothing
    
    ref_pct = ref_counts / ref_counts.sum()
    cur_pct = cur_counts / cur_counts.sum()
    
    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return psi

# --- Simulate monitoring: compare train vs test distributions ---
print("📊 DRIFT MONITORING REPORT")
print("="*60)
print("Comparing TRAINING data vs SCORING data (test set as proxy)")
print()

monitored_features = NUMERIC_FEATURES
psi_results = []

for feat in monitored_features:
    psi_val = calculate_psi(X_train[feat], X_test[feat])
    status = "✅ Stable" if psi_val < 0.1 else ("⚠️ Moderate" if psi_val < 0.2 else "🚨 DRIFT")
    psi_results.append({'Feature': feat, 'PSI': psi_val, 'Status': status})

psi_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
print("Population Stability Index (PSI):")
print("  PSI < 0.1  : No significant drift")
print("  0.1 ≤ PSI < 0.2 : Moderate shift, monitor closely")
print("  PSI ≥ 0.2  : Significant drift, investigate/retrain")
print()
display(psi_df)

# --- Visualize drift for top features ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(monitored_features[:4]):
    ax = axes[i]
    ax.hist(X_train[feat], alpha=0.6, bins=30, label='Training', color='steelblue', density=True)
    ax.hist(X_test[feat], alpha=0.6, bins=30, label='Scoring', color='coral', density=True)
    psi_val = psi_df[psi_df['Feature']==feat]['PSI'].values[0]
    ax.set_title(f'{feat} (PSI={psi_val:.4f})', fontsize=11)
    ax.legend()
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')

plt.suptitle('Feature Distribution: Training vs Scoring', fontsize=14)
plt.tight_layout()
plt.show()

# --- Prediction drift ---
train_preds = best_model.predict_proba(X_train)[:, 1]
test_preds = best_model.predict_proba(X_test)[:, 1]
pred_psi = calculate_psi(pd.Series(train_preds), pd.Series(test_preds))

print(f"\n🎯 Prediction Drift (PSI): {pred_psi:.4f}")
if pred_psi < 0.1:
    print("   ✅ No significant prediction drift detected")
elif pred_psi < 0.2:
    print("   ⚠️ Moderate prediction drift — schedule monitoring review")
else:
    print("   🚨 Significant prediction drift — trigger retraining!")

print("\n" + "="*60)
print("📋 MONITORING RECOMMENDATIONS:")
print("  1. Schedule PSI computation weekly on new scored claims")
print("  2. Set alert threshold: PSI > 0.2 on any feature")
print("  3. Monthly model performance review on labelled samples")
print("  4. Quarterly full retrain regardless of drift signals")
print("  5. Model retirement: if F1 < 0.60 for 2 consecutive weeks")

## Lakehouse Monitoring

We attach a **Databricks Lakehouse Monitor** (InferenceLog type) to the scored predictions table. This provides:
- Automated drift detection on features and predictions
- Model quality metrics (once ground-truth labels are joined)
- Time-windowed statistics with configurable granularity
- Dashboard and alerting integration

In [0]:
# ============================================================
# LAKEHOUSE MONITORING - InferenceLog Monitor
# ============================================================
import requests
import json

MONITORED_TABLE = "zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions"
OUTPUT_SCHEMA = "zurich_lcnc_rfp_catalog.pc_demo"
USER_EMAIL = "stephan.wernz@databricks.com"
ASSETS_DIR = f"/Workspace/Users/{USER_EMAIL}/databricks_lakehouse_monitoring/{MONITORED_TABLE}"

print(f"Creating Lakehouse Monitor on: {MONITORED_TABLE}")
print("=" * 50)
print(f"   Type: InferenceLog (Classification)")
print(f"   Prediction column: predicted_high_severity")
print(f"   Timestamp column: scored_at")
print(f"   Model ID column: model_version")
print(f"   Granularities: 1 day")
print(f"   Output schema: {OUTPUT_SCHEMA}")
print()

# Get auth context from notebook
host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)

if not host.startswith("http"):
    host = f"https://{host}"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Create monitor via REST API
payload = {
    "assets_dir": ASSETS_DIR,
    "output_schema_name": OUTPUT_SCHEMA,
    "inference_log": {
        "problem_type": "PROBLEM_TYPE_CLASSIFICATION",
        "prediction_col": "predicted_high_severity",
        "timestamp_col": "scored_at",
        "model_id_col": "model_version",
        "granularities": ["1 day"]
    }
}

resp = requests.post(
    f"{host}/api/2.1/unity-catalog/tables/{MONITORED_TABLE}/monitor",
    headers=headers,
    json=payload
)

if resp.status_code == 200:
    result = resp.json()
    print("Lakehouse Monitor created successfully!")
    print(f"   Status: {result.get('status', 'active')}")
    print(f"   Dashboard ID: {result.get('dashboard_id', 'pending')}")
elif resp.status_code == 409 or "ALREADY_EXISTS" in resp.text:
    print("Monitor already exists on this table.")
    get_resp = requests.get(
        f"{host}/api/2.1/unity-catalog/tables/{MONITORED_TABLE}/monitor",
        headers=headers
    )
    if get_resp.status_code == 200:
        info = get_resp.json()
        print(f"   Status: {info.get('status', 'unknown')}")
        print(f"   Dashboard ID: {info.get('dashboard_id', 'N/A')}")
else:
    print(f"Response ({resp.status_code}): {resp.text[:500]}")

print(f"\nMonitored table: {MONITORED_TABLE}")
print(f"\nOutput tables (auto-created after first refresh):")
print(f"   Profile metrics: {MONITORED_TABLE}_profile_metrics")
print(f"   Drift metrics:   {MONITORED_TABLE}_drift_metrics")
print(f"\nNote: Once ground-truth labels are joined (column: 'label'),")
print(f"the monitor will also compute precision, recall, F1, and AUC.")

In [0]:
# ============================================================
# REFRESH MONITOR (trigger initial metric computation)
# ============================================================

print("Triggering monitor refresh...")
print("=" * 50)

try:
    refresh_resp = requests.post(
        f"{host}/api/2.1/unity-catalog/tables/{MONITORED_TABLE}/monitor/refreshes",
        headers=headers
    )
    
    if refresh_resp.status_code == 200:
        refresh_info = refresh_resp.json()
        print(f"Refresh triggered!")
        print(f"   Refresh ID: {refresh_info.get('refresh_id', 'running')}")
        print(f"   State: {refresh_info.get('state', 'PENDING')}")
    else:
        print(f"Refresh response ({refresh_resp.status_code}): {refresh_resp.text[:300]}")

    print(f"\nRefresh runs asynchronously. Check status in Catalog Explorer:")
    print(f"   Navigate to: {MONITORED_TABLE} > Quality tab")
    print(f"\nOnce complete, query metrics:")
    print(f"   SELECT * FROM {MONITORED_TABLE}_profile_metrics")
    print(f"   SELECT * FROM {MONITORED_TABLE}_drift_metrics")

except Exception as e:
    print(f"Refresh note: {e}")
    print("\nYou can trigger a refresh manually from the Quality tab in Catalog Explorer.")

In [0]:
%sql
-- ============================================================
-- QUERY LAKEHOUSE MONITOR METRICS
-- ============================================================
-- Profile metrics (statistics per time window) - populated after first refresh
SELECT
  window,
  granularity,
  column_name,
  data_type,
  model_version,
  count,
  avg,
  percent_null,
  percent_zeros,
  percent_distinct
FROM zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions_profile_metrics
WHERE column_name IN ('predicted_high_severity', 'severity_probability', 'premium_amount', 'risk_score')
ORDER BY column_name, window.start DESC
LIMIT 20

In [0]:
# ============================================================
# GENERATE HISTORICAL SCORING RUNS FOR MONITOR
# ============================================================
# To populate drift metrics, we need predictions across multiple
# time windows. We'll score subsets of data with backdated timestamps
# to simulate daily scoring runs over the past week.
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import numpy as np

print("Generating historical scoring runs to populate monitor...")
print("=" * 60)

# Load the source data
source_df = spark.table("zurich_lcnc_rfp_catalog.pc_demo.curated_claims_analytics_designer")
pdf = source_df.toPandas()

# Prepare features (same as training pipeline)
CATEGORICAL_FEATURES = ['policy_type', 'region']
NUMERIC_FEATURES = ['premium_amount', 'coverage_limit', 'deductible', 'risk_score',
                    'reporting_delay_days', 'claim_month', 'claim_year']

features_for_scoring = pd.get_dummies(pdf[CATEGORICAL_FEATURES + NUMERIC_FEATURES], 
                                       columns=CATEGORICAL_FEATURES, drop_first=True)

# Align columns with training data
missing_cols = set(X_train.columns) - set(features_for_scoring.columns)
for col in missing_cols:
    features_for_scoring[col] = 0
features_for_scoring = features_for_scoring[X_train.columns]

# Generate predictions for 5 historical days
days_back = [7, 5, 3, 2, 1]  # Days in the past
rows_per_day = 100000  # 100K per day for variety

all_scored_dfs = []

for days_ago in days_back:
    scored_at = datetime.now() - timedelta(days=days_ago)
    
    # Sample a subset (with replacement for variety)
    np.random.seed(42 + days_ago)  # Different seed per day
    sample_idx = np.random.choice(len(pdf), size=rows_per_day, replace=False)
    
    sample_features = features_for_scoring.iloc[sample_idx]
    sample_source = pdf.iloc[sample_idx]
    
    # Predict
    preds = best_model.predict(sample_features)
    probs = best_model.predict_proba(sample_features)[:, 1]
    
    # Build scored DataFrame
    scored_pdf = pd.DataFrame({
        'claim_id': sample_source['claim_id'].values,
        'policy_id': sample_source['policy_id'].values,
        'policy_type': sample_source['policy_type'].values,
        'region': sample_source['region'].values,
        'risk_score': sample_source['risk_score'].values,
        'premium_amount': sample_source['premium_amount'].values,
        'coverage_limit': sample_source['coverage_limit'].values,
        'deductible': sample_source['deductible'].values,
        'predicted_high_severity': preds.astype(int),
        'severity_probability': probs,
        'scored_at': scored_at,
        'model_version': 'Random Forest'
    })
    
    all_scored_dfs.append(scored_pdf)
    print(f"   Day -{days_ago} ({scored_at.strftime('%Y-%m-%d')}): {rows_per_day:,} predictions scored")

# Combine all historical runs
historical_pdf = pd.concat(all_scored_dfs, ignore_index=True)
print(f"\nTotal historical predictions: {len(historical_pdf):,}")

# Write to the predictions table (append)
historical_sdf = spark.createDataFrame(historical_pdf)
historical_sdf.write.mode("append").saveAsTable(
    "zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions"
)

print(f"\n✅ Appended {len(historical_pdf):,} historical predictions to table.")
print(f"\nTime windows now available:")
window_counts = spark.sql("""
    SELECT DATE(scored_at) as scoring_date, COUNT(*) as predictions
    FROM zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions
    GROUP BY DATE(scored_at)
    ORDER BY scoring_date
""").toPandas()
for _, row in window_counts.iterrows():
    print(f"   {row['scoring_date']}: {row['predictions']:,} predictions")

print(f"\nTotal windows: {len(window_counts)} (drift metrics need 2+)")

# Trigger monitor refresh
print("\n" + "=" * 60)
print("Triggering monitor refresh to compute metrics across all windows...")

refresh_resp = requests.post(
    f"{host}/api/2.1/unity-catalog/tables/{MONITORED_TABLE}/monitor/refreshes",
    headers=headers
)

if refresh_resp.status_code == 200:
    refresh_info = refresh_resp.json()
    print(f"   Refresh triggered! ID: {refresh_info.get('refresh_id', 'running')}")
    print(f"   This will take 2-5 minutes to complete.")
    print(f"\n   Once done, the monitoring dashboard will show:")
    print(f"   - Feature distributions per time window")
    print(f"   - Drift metrics (KS test, PSI, Chi-squared)")
    print(f"   - Performance trends over time")
    print(f"   - Prediction distribution changes")
else:
    print(f"   Refresh response ({refresh_resp.status_code}): {refresh_resp.text[:300]}")

## Demo Complete!

### Summary of what we demonstrated:

| Step | What | Platform Feature |
| --- | --- | --- |
| 1 | Data loading & EDA | Unity Catalog + Spark |
| 2 | Target & leakage analysis | Domain expertise |
| 3 | Feature preparation | Pandas + stratified split |
| 4 | AutoML + manual model comparison | `databricks.automl` + scikit-learn |
| 5 | Evaluation (ROC, confusion matrix, F1) | MLflow metric logging |
| 6 | Explainability | Feature importance + SHAP |
| 7 | Experiment tracking | MLflow experiments |
| 8 | Model registration | Unity Catalog model registry |
| 9 | Feature Store | Feature Engineering in Unity Catalog |
| 10 | Batch scoring (Feature Store) | `fe.score_batch()` with auto-lookup |
| 11 | Real-time serving endpoint | Model Serving (scale-to-zero) |
| 12 | A/B testing (champion vs challenger) | Traffic split routing |
| 13 | Inference logging & route comparison | AI Gateway inference tables |
| 14 | Drift detection (PSI) | Custom monitoring code |
| 15 | Lakehouse Monitoring | Automated drift & performance tracking |
| 16 | Scheduled batch scoring | Lakeflow Jobs (weekly) |

### Key Artefacts Created:
- **Model Registry:** `zurich_lcnc_rfp_catalog.pc_demo.high_severity_claim_model` (champion v1, challenger v3)
- **Feature Table:** `zurich_lcnc_rfp_catalog.pc_demo.claims_feature_store` (13 features, claim_id PK)
- **Serving Endpoint:** `high-severity-claims-endpoint` (80/20 A/B split)
- **Predictions Table:** `zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions` (1.1M rows, 6 time windows)
- **Lakehouse Monitor:** InferenceLog monitor with daily granularity, auto-computed accuracy/F1/precision/recall
- **Scheduled Job:** `Weekly Claims Severity Batch Scoring` (Mondays 07:00 Europe/Zurich)

### Next Steps (beyond this demo):
- Promote challenger to champion once A/B results validate improvement
- Add alerting rules on drift p-values and F1 degradation
- Integrate with claims management system via REST API
- Add data quality checks (UC constraints or expectations)
- Configure online feature store for real-time serving lookups